# 📊 Matplotlib Tutorial 3: Box Plots, Violin Plots & Statistical Visualizations

**Objective:** Master statistical plots for data analysis

In this notebook, you'll learn:
- Box plots for distribution summaries
- Violin plots for density visualization
- Error bars and confidence intervals
- Statistical comparisons across categories

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load Titanic dataset
df = pd.read_csv('../titanic.csv')
df.head()

## 1. Box Plots

Box plots show the five-number summary: min, Q1, median, Q3, max

In [ ]:
# Basic box plot - Fare by passenger class
fig, ax = plt.subplots(figsize=(10, 6))

data_by_class = [df[df['Pclass'] == c]['Fare'] for c in [1, 2, 3]]

bp = ax.boxplot(data_by_class, labels=['1st Class', '2nd Class', '3rd Class'],
                patch_artist=True, notch=True)

# Customize colors
colors = ['#FFD700', '#C0C0C0', '#CD7F32']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xlabel('Passenger Class', fontsize=14)
ax.set_ylabel('Fare ($)', fontsize=14)
ax.set_title('Fare Distribution by Class - Box Plot', fontsize=16, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.show()

In [ ]:
# Horizontal box plot
fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot(data_by_class, labels=['1st Class', '2nd Class', '3rd Class'],
                patch_artist=True, vert=False)

colors = ['#FFD700', '#C0C0C0', '#CD7F32']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xlabel('Fare ($)', fontsize=14)
ax.set_ylabel('Passenger Class', fontsize=14)
ax.set_title('Fare Distribution by Class - Horizontal Box Plot', fontsize=16, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.show()

In [ ]:
# Multiple box plots - Age by survival and class
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data
data = []
labels = []
for survived in [0, 1]:
    for pclass in [1, 2, 3]:
        subset = df[(df['Survived'] == survived) & (df['Pclass'] == pclass)]['Age'].dropna()
        data.append(subset)
        labels.append(f'{"Died" if survived == 0 else "Survived"}\nPclass {pclass}')

bp = ax.boxplot(data, labels=labels, patch_artist=True)

# Color by survival status
colors = ['#FF6B6B', '#FF6B6B', '#4ECDC4', '#4ECDC4', '#45B7D1', '#45B7D1']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xlabel('Survival Status & Class', fontsize=14)
ax.set_ylabel('Age', fontsize=14)
ax.set_title('Age Distribution by Survival and Class', fontsize=16, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Violin Plots

Violin plots combine box plots with kernel density estimation

In [ ]:
# Basic violin plot
fig, ax = plt.subplots(figsize=(10, 6))

vp = ax.violinplot(data_by_class, positions=[1, 2, 3], showmeans=True, showmedians=True)

# Customize colors
colors = ['#FFD700', '#C0C0C0', '#CD7F32']
for pc, color in zip(vp['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
ax.set_xlabel('Passenger Class', fontsize=14)
ax.set_ylabel('Fare ($)', fontsize=14)
ax.set_title('Fare Distribution by Class - Violin Plot', fontsize=16, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.show()

## 3. Error Bars

Show uncertainty or variability in measurements

In [ ]:
# Calculate mean and standard error
class_stats = df.groupby('Pclass')['Fare'].agg(['mean', 'std', 'count'])
class_stats['se'] = class_stats['std'] / np.sqrt(class_stats['count'])

fig, ax = plt.subplots(figsize=(10, 6))

x = class_stats.index
y = class_stats['mean']
yerr = class_stats['se']

ax.bar(x, y, yerr=yerr, capsize=10, color=['#FFD700', '#C0C0C0', '#CD7F32'],
       edgecolor='black', alpha=0.7)

ax.set_xlabel('Passenger Class', fontsize=14)
ax.set_ylabel('Mean Fare ($)', fontsize=14)
ax.set_title('Mean Fare by Class with Standard Error', fontsize=16, fontweight='bold')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['1st', '2nd', '3rd'])
ax.grid(axis='y', alpha=0.3)

plt.show()

## 4. Line Plots with Multiple Variables

In [ ]:
# Survival rate by age groups
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 10, 20, 30, 40, 50, 60, 70, 80], 
                        labels=['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80'])

survival_by_age = df.groupby('AgeGroup')['Survived'].mean()

fig, ax1 = plt.subplots(figsize=(10, 6))

# Line plot for survival rate
color = 'tab:blue'
ax1.set_xlabel('Age Group', fontsize=14)
ax1.set_ylabel('Survival Rate', color=color, fontsize=14)
ax1.plot(survival_by_age.index, survival_by_age.values, color=color, linewidth=2, 
         marker='o', markersize=8, label='Survival Rate')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 1)

# Create second y-axis for passenger count
ax2 = ax1.twinx()
age_counts = df['AgeGroup'].value_counts().sort_index()
color = 'tab:red'
ax2.set_ylabel('Passenger Count', color=color, fontsize=14)
ax2.bar(age_counts.index, age_counts.values, color=color, alpha=0.3)
ax2.tick_params(axis='y', labelcolor=color)

ax1.set_title('Survival Rate and Passenger Count by Age Group', fontsize=16, fontweight='bold')

# Legend
lines1, labels1 = ax1.get_legend_handles_labels()
ax1.legend(lines1, labels1, loc='upper left')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Stack Plots

In [ ]:
# Stacked bar chart - Survival by class and gender
survival_by_class_gender = df.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(3)
width = 0.35

bars1 = ax.bar(x - width/2, survival_by_class_gender['female'], width, 
               label='Female', color='#FF69B4', edgecolor='black')
bars2 = ax.bar(x + width/2, survival_by_class_gender['male'], width, 
               label='Male', color='#4169E1', edgecolor='black')

ax.set_xlabel('Passenger Class', fontsize=14)
ax.set_ylabel('Survival Rate', fontsize=14)
ax.set_title('Survival Rate by Class and Gender', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['1st', '2nd', '3rd'])
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)

plt.show()

## 6. Heatmaps (Correlation Matrix)

In [ ]:
# Calculate correlation matrix
numeric_df = df[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']].dropna()
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))

# Create heatmap
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', aspect='auto')

# Set ticks
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, fontsize=12)
ax.set_yticklabels(corr_matrix.columns, fontsize=12)

# Add correlation values
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        text = ax.text(j, i, f'{corr_matrix.values[i, j]:.2f}',
                      ha='center', va='center', color='black', fontsize=11)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation Coefficient', fontsize=12)

ax.set_title('Correlation Matrix - Titanic Dataset', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## 7. Custom Colorbars and Colormaps

In [ ]:
# Demonstrate different colormaps
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

colormaps = ['viridis', 'plasma', 'inferno', 'magma', 'cividis', 'RdBu_r']

for ax, cmap in zip(axes.flat, colormaps):
    # Create sample data
    x = np.linspace(0, 10, 100)
    y = np.linspace(0, 10, 100)
    X, Y = np.meshgrid(x, y)
    Z = np.sin(X) * np.cos(Y)
    
    im = ax.imshow(Z, cmap=cmap, aspect='auto')
    ax.set_title(cmap, fontsize=14, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 8. Key Takeaways

✅ **Box plots** show distribution summaries and outliers  
✅ **Violin plots** combine box plots with density estimation  
✅ **Error bars** represent uncertainty in measurements  
✅ **Dual y-axes** compare different scales  
✅ **Heatmaps** visualize correlation matrices  
✅ **Colormaps** enhance data interpretation  

---

**Next:** Tutorial 4 - Real-World EDA Project with Matplotlib